<img src="https://portal.ombiel.co.uk/assets/MiddlesexUniversity/Middlesex/Content_Migration_IMAGES/tiles-icons/tiles-without-labels/student-life/student-lifeMy-MDX-app.png" class="logo me-1" alt="mdx" width=100>

<h1> Introduction to Climate Data Analysis</h1>

This coursework focuses on the multidimensional analysis of climate data for `New York City`.

The dataset used in this study was obtained from the `Global Historical Climate Network (GHCN)`, a publicly available database that provides daily land surface climate observations. The data were collected from the weather station NY City Central Park, NY, US (Station ID: `GHCND:USW00094728`). 

Prior to conducting the analysis, a data cleaning and preprocessing stage is undertaken to ensure the reliability and consistency of the dataset.

Data Source (Downloaded): https://www.ncei.noaa.gov/cdo-web/datasets <br/>

<h2> Table of Contents</h2>
<div class="alert alert-info">  

1. Data Analysis and Pre-processing   
2. Insight Extraction (Discover information / present data)  
    - Average temperature trend (monthly / yearly)
    - Finding meteorological anomalies (Hurricane / Flood events in recent years)
    - Relationship between seasonal temperature and electricity demand
    - Comparing the weather conditions (1990s and 2020s)
3. Story Telling / Presentation

</div>
<hr>

<h1>Data Analysis and Pre-processing </h1>
Initially, the dataset is analysed and pre-processed to ensure data quality and consistency before further analysis.

Load the weather data from local Mongo DB, which is imported datasets of Daily Summaries (1995-2000 and 2020-2025) from central park station.

In [1]:
from pymongo import MongoClient
import pandas as pd
import altair as alt
from altair.datasets import data

try:
    # connect to local MongoDB
    client = MongoClient("mongodb://localhost:27017/")

    # access database and collection
    db = client["cst_4068"]
    weather = db["noaa_weather_centralpark"]
   
    # query (select all)
    query = {}
    w_results = weather.find(query)
  
    # convert cursor to dataframe
    wdf = pd.DataFrame(list(w_results))
  
    # clean dataframe (drop the '_id' column so that Atair can read it)
    if not wdf.empty:        
        if '_id' in wdf.columns:
            wdf = wdf.drop(columns=['_id'])  
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    if 'client' in locals():
        client.close()

View the Dataframe info

In [2]:
wdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4018 entries, 0 to 4017
Data columns (total 31 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   STATION    4018 non-null   object 
 1   NAME       4018 non-null   object 
 2   LATITUDE   4018 non-null   float64
 3   LONGITUDE  4018 non-null   float64
 4   ELEVATION  4018 non-null   float64
 5   DATE       4018 non-null   object 
 6   AWND       3676 non-null   float64
 7   PRCP       4018 non-null   float64
 8   SNOW       4018 non-null   float64
 9   SNWD       3277 non-null   float64
 10  TMAX       4018 non-null   float64
 11  TMIN       4018 non-null   float64
 12  WT01       1421 non-null   float64
 13  WT08       742 non-null    float64
 14  WT03       155 non-null    float64
 15  TSUN       321 non-null    float64
 16  WT02       138 non-null    float64
 17  WT04       12 non-null     float64
 18  WT06       26 non-null     float64
 19  WT05       2 non-null      float64
 20  TAVG    

In [3]:
print(f"Rows and columns count: {wdf.shape}")

Rows and columns count: (4018, 31)


Standardize attributes name to lower case

In [4]:
wdf.columns = wdf.columns.str.lower()

Rename attributes to have readability <br/>
<small>Ref: https://www.ncei.noaa.gov/data/daily-summaries/doc/GHCND_documentation.pdf</small>

In [5]:
rename_map = {
    'tmax': 'temp_max',
    'tmin': 'temp_min',
    'tavg': 'temp_avg',
    'prcp': 'precipitation',
    'awnd': 'wind_speed',
    'snow': 'snowfall',
    'snwd': 'snow_depth',
    'tsun': 'sunshine_duration',
    'wt01': 'wt_fog',
    'wt02': 'wt_heavy_fog',
    'wt03': 'wt_thunder',
    'wt04': 'wt_ice_pellets',
    'wt05': 'wt_hail',
    'wt06': 'wt_glaze',
    'wt07': 'wt_dust_ash',      # found in older 90s records
    'wt08': 'wt_smoke_haze',
    'wt09': 'wt_blowing_snow',  # important for the 1996 Blizzard
    'wt11': 'wt_high_winds',    # common in 90s storm data
    'wt13': 'wt_mist',
    'wt14': 'wt_drizzle',
    'wt16': 'wt_rain',          # historical specific code
    'wt18': 'wt_snow'           # historical specific code
}

wdf = wdf.rename(columns=rename_map)

Change object type to datetime for `date` attribute, and create month, year columns

In [6]:
wdf['date'] = pd.to_datetime(wdf['date'], dayfirst=True)
wdf['month'] = wdf['date'].dt.month
wdf['year'] = wdf['date'].dt.year

print(wdf['date'].dtype)

datetime64[ns]


Drop unnessary attributes; drop sunshine_duration and temp_avg attributes as entire columns are empty

In [7]:
wdf = wdf.drop(columns= ['station', 'name', 'latitude', 'longitude', 'elevation', \
        'snow_depth', 'sunshine_duration', 'temp_avg', 'wt15', 'wt17', 'wt19', 'wt21', 'wt22'])

Filter the years for data analysis: <mark> late 1990s (1996–1999) and the early 2020s (2021–2024). </mark>

In [8]:
wdf = wdf[wdf['year'].isin([1996, 1997, 1998, 1999, 2021, 2022, 2023, 2024])]
wdf = wdf.reset_index(drop=True)

Handeling missing value

In [9]:
wdf.isnull().sum()

date                 0
wind_speed          26
precipitation        0
snowfall             0
temp_max             0
temp_min             0
wt_fog            1762
wt_smoke_haze     2302
wt_thunder        2810
wt_heavy_fog      2806
wt_ice_pellets    2916
wt_glaze          2906
wt_hail           2921
wt_rain           2321
wt_snow           2824
wt_drizzle        2922
wt_mist           2393
wt_high_winds     2920
month                0
year                 0
dtype: int64

In NOAA data, these columns only contain a 1 if the event (such as Fog, Thunder, etc.) happened. If it didn't happen, the cell is left empty (NaN). <br/>
Thus, replace with 0 for Nan values of `wt_*` (weather type columns)
- 1 is Yes
- 0 is No

In [10]:
wt_cols = [col for col in wdf.columns if col.startswith('wt_')]

wdf[wt_cols] = wdf[wt_cols].fillna(0)

Handling Missing Wind Speed Data (Continuous Data / Measurements)

Based on Temporal autocorrelation assumption that past values actively influence present values. For example today's temperature often depends strongly on yesterday's, exhibiting clear temporal correlation and persistence. <small>(Ref from CST_4075, Week 8 Temporal Data)</small>

Missing values in wind_speed is estimated based on neighbouring observations and preserves local temporal trends better than mean imputation. The parameter limit_direction='both' ensures gaps at the start or end of the series are also filled.

In [11]:
wdf['wind_speed'] = wdf['wind_speed'].interpolate(method='linear', limit_direction='both')

Recheck null value columns.

In [12]:
wdf.isnull().sum()

date              0
wind_speed        0
precipitation     0
snowfall          0
temp_max          0
temp_min          0
wt_fog            0
wt_smoke_haze     0
wt_thunder        0
wt_heavy_fog      0
wt_ice_pellets    0
wt_glaze          0
wt_hail           0
wt_rain           0
wt_snow           0
wt_drizzle        0
wt_mist           0
wt_high_winds     0
month             0
year              0
dtype: int64

Check thunderstorm attribute.

In [13]:
wdf[wdf['wt_thunder'].isin([1])].groupby('year').size().reset_index(name='thunder_count')

,year,thunder_count
0,2021,30
1,2022,25
2,2023,34
3,2024,23


One notable issue is that thunder (`wt_thunder`, renamed from `wt03`) is entirely unrecorded for 1996-1999, although observations are available for 2021–2024.

The absence of WT03 (thunder) data at Central Park from 1996 to 1999 is a known historical gap caused by the modernisation of the weather station during that period.

This gap arose because, in late 1995, the National Weather Service transitioned the Central Park station to an Automated Surface Observing System (ASOS). On 31 December 1995, the final human observers left the site, and observations became fully automated.

However, due to sensor limitations, early ASOS systems were not equipped with automated lightning or thunder detection. Consequently, for several years, certain weather types (such as thunder) went unrecorded, as there were neither human observers nor suitable automated sensors.

Automated lightning sensors were only installed in the Central Park ASOS system around 2005–2006.

Therefore, thunder data from the nearby LaGuardia Airport station (LGA; Station ID: `USW00014732`, approximately 6 miles away) were used for the period 1996–1999 and combined into the current dataset, enabling comparison between the 1990s (1996–1999) and the 2020s (2021–2024).

<small>Ref: https://vlab.noaa.gov/web/nws-heritage/-/100-years-of-weather-observations-at-belvedere-castle-in-central-park<small>

In [14]:
try:   
    client = MongoClient("mongodb://localhost:27017/")

    # noaa_weather_lga collection with 1996-1999 records 
    # from stationid USW00014732 for thunderstorm data    
    db = client["cst_4068"]
    lga = db["noaa_weather_lga"]  
    
    # fetch date and thunderstorm columns only
    query = {}
    projection = {
        "DATE": 1, 
        "WT03": 1,      
        "_id": 0
    }
    results = lga.find(query, projection)  
    lga = pd.DataFrame(list(results))      
    
    rename_map = {
        'DATE': 'date',
        'WT03': 'wt_thunder',   
    }

    lga = lga.rename(columns=rename_map)
    lga['wt_thunder'].fillna(0)
 
except Exception as e:
    print(f"An error occurred: {e}")

finally:
    if 'client' in locals():
        client.close()

Add 90s's thunder data into the current grand master dataset and verify.

In [15]:
wdf['date'] = pd.to_datetime(wdf['date']).dt.normalize()
lga['date'] = pd.to_datetime(lga['date']).dt.normalize()

wdf = pd.merge(wdf, lga[['date', 'wt_thunder']], on='date', how='left', suffixes=('', '_lga_data'))

mask = (wdf['date'].dt.year >= 1996) & (wdf['date'].dt.year <= 1999)

wdf.loc[mask, 'wt_thunder'] = wdf.loc[mask, 'wt_thunder_lga_data']

#wdf['wt_thunder'] = wdf['wt_thunder'].fillna(wdf['wt_thunder_lga_data'])
wdf['wt_thunder'] = wdf['wt_thunder'].fillna(0)

wdf.drop(columns=['wt_thunder_lga_data'], inplace=True)

wdf[wdf['wt_thunder'].isin([1])].groupby('year').size().reset_index(name='thunder_count')

,year,thunder_count
0,1996,21
1,1997,19
2,1998,27
3,1999,23
4,2021,30
5,2022,25
6,2023,34
7,2024,23


Check whether duplicate items.

In [16]:
print(f"Duplicate rows count: {wdf.duplicated().sum()}")

Duplicate rows count: 0


Add month_name in order to have readability for month number 1,2,3,etc.

In [17]:
import calendar
pd.options.display.max_columns = None

month_map = {i: calendar.month_abbr[i] for i in range(1, 13)}
wdf['month_name'] = wdf['month'].map(month_map)

Finally, review the dataset randomly.

In [18]:
print(f"Rows and columns count: {wdf.shape}")
print(f"View data for random rows:")
wdf.sample(20)

Rows and columns count: (2922, 21)
View data for random rows:


,date,wind_speed,precipitation,snowfall,temp_max,temp_min,wt_fog,wt_smoke_haze,wt_thunder,wt_heavy_fog,wt_ice_pellets,wt_glaze,wt_hail,wt_rain,wt_snow,wt_drizzle,wt_mist,wt_high_winds,month,year,month_name
1266,2024-06-20,1.5,0.0,0.0,32.8,22.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6,2024,Jun
2328,1998-05-17,2.6,0.0,0.0,19.4,14.4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1998,May
1523,1996-03-03,5.6,0.3,0.0,1.7,-6.1,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,3,1996,Mar
1470,1996-01-10,4.4,0.0,0.0,-0.6,-6.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1996,Jan
1305,2024-07-29,1.3,1.3,0.0,28.9,20.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7,2024,Jul
372,2022-01-08,2.7,0.0,0.0,-1.0,-6.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,2022,Jan
2372,1998-06-30,2.9,20.1,0.0,27.8,18.3,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,6,1998,Jun
914,2023-07-04,0.3,16.0,0.0,28.3,22.8,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7,2023,Jul
1471,1996-01-11,2.7,0.0,0.0,-3.3,-8.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1996,Jan
1167,2024-03-13,1.6,0.0,0.0,16.7,8.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,2024,Mar


<hr>

<h1>Insight Extraction (Discover information / present data)</h1> 

In this section, New York City weather data are analysed to identify trends, comparisons, correlations, and anomalies. The analysis includes:

- Average temperature trend (monthly / yearly)
- Finding meteorological anomalies (Hurricane / Flood events in recent years)
- Relationship between seasonal temperature and electricity demand
- Comparing the weather conditions (1990s and 2020s)

<h2>Average Temperature Trend (Monthly / Yearly) </h2>

Monthly temperature trends in New York City are analysed by comparing the periods 1996–1999 and 2021–2024. 

In [19]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# interactive selection for legend
legend_select = alt.selection_point(
    fields=["year"],
    bind="legend",
    toggle=True
)

# create the line chart for monthly averages
alt.Chart(wdf).mark_line(point=True).encode(
    x=alt.X('month_name:O', title='Month', sort=month_order, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('mean(temp_max):Q', title='Average Monthly Temperature (temp_max °C)'),
    color=alt.Color(
        'year:N',
        legend = alt.Legend(
            title='Year',
            orient='none',
            legendX=730, 
            legendY=10,
            direction='vertical',
            fillColor='white',
            strokeColor='black',
            cornerRadius=7,
            padding=7,
            offset=0
        )
    ),
    opacity=alt.condition(legend_select, alt.value(1), alt.value(0.15)),
    tooltip=[
        alt.Tooltip('year:N', title='Year'),
        alt.Tooltip('month:O', title='Month'),
        alt.Tooltip('mean(temp_max):Q', format='.2f', title='Avg Month Temp (°C)')
    ]
).add_params(legend_select
).properties(
    title={
        "text": 'NYC Monthly Average Temperature Trends', 
        "fontSize": 15,
        "subtitle": ["1996-1999 vs 2021-2024"],
        "subtitleFontSize": 12
    },
    width=800,
    height=350,
    padding={'left': 10, 'top': 10, 'right': 10, 'bottom': 10}
).display()

alt.Chart(...)

According to the graph, generally monthly temperature trends have not changed significantly 25 years ago and recent years. Temperatures drop during the winter (January, February, and December) and jump during the summer (June, July, and August).

While the overall temperature cycle remains similar, the annual temperature trend is examined further below.

In [20]:
# prepare yearly data
yearly_avg = wdf.groupby('year')['temp_max'].mean().reset_index()

# base chart configuration
base = alt.Chart(yearly_avg).encode(
    x=alt.X('year:N', title='Year', axis=alt.Axis(labelAngle=0))
)

# create the bar chart for yearly averages
bars = base.mark_bar(
    cornerRadiusTopLeft=3, 
    cornerRadiusTopRight=3
).encode(
    y=alt.Y('temp_max:Q', 
            title='Average Yearly Temperature (temp_max °C)', 
            scale=alt.Scale(scheme='tableau10')),
    color=alt.Color('year:N', scale=alt.Scale(scheme='tableau10'), legend=None), 
    tooltip=[
        alt.Tooltip('year:N', title='Year'),
        alt.Tooltip('temp_max:Q', format='.2f', title='Avg Yearly Temperature (°C)')
    ]
)

# text label (static values on top of bars)
text = base.mark_text(
    align='center',
    baseline='bottom',
    dy=-2, 
    fontSize=11,
    fontWeight='bold'
).encode(
    y=alt.Y('temp_max:Q'),
    text=alt.Text('temp_max:Q', format='.2f') 
)

# display (layer and bar)
(bars + text).properties(
    title={
        "text": 'NYC Annual Average Temperature Trends', 
        "fontSize": 15,
        "subtitle": ["1996-1999 vs 2021-2024"],
        "subtitleFontSize": 12
    },
    width=550,
    height=300,
    padding={'left': 5, 'top': 5, 'right': 20, 'bottom': 5} 
).display()

alt.LayerChart(...)

The yearly average temperature analysis reveals a gradual upward trend in NYC's thermal baseline. The annual average day temperature rose from 15.84°C in 1996 to 18.32°C in 2024, marking a total increase of 2.48°C across the study periods.

However, the data are not temporally continuous (25 years gap). The warming pattern is not strictly linear; for instance, 1998 (17.84°C) was warmer than 2021 (17.68°C). Despite year-to-year variability, there is an overall upward tendency in temperature, with the most recent years (2023–2024) establishing the highest recorded values in the dataset.


<h2>Finding Meteorological Anomalies (Hurricane / Flood events in recent years)</h2>

Hurricanes and floods are usually extreme values in Wind speed and Precipitation. To identify such events, Z-scores will be calculated for both variables. Z-scores quantify the extent to which observations deviate from the standard, so that significant anomalies detect in wind speed and precipitation. This analysis focuses for the year 2021–2024.

In [21]:
from scipy.stats import zscore

recent_df = wdf[wdf['year'] >= 2021].copy()

# z-score formula: z = (value - mean) / standard_deviation
recent_df['wind_z'] = zscore(recent_df['wind_speed'])
recent_df['precip_z'] = zscore(recent_df['precipitation'])

# high wind OR high rain (z-score > 2), where 2 is a common threshold 
# z-score > 2 means that, identifying the outliers 
# that are more than 2 standard deviations above the mean
# filter with (wind_z > 2 | precip_z > 2) instead of (wind_z > 2 & precip_z > 2) 
# to capture storms with either extreme wind or rain
storms_df = recent_df[
    (recent_df['wind_z'] > 2) | 
    (recent_df['precip_z'] > 2)
].sort_values('precipitation', ascending=False)

# view the few extreme results
storms_df[['date', 'precipitation', 'wind_speed', 'wind_z', 'precip_z']].head(10)

,date,precipitation,wind_speed,wind_z,precip_z
243,2021-09-01,181.1,3.8,1.534629,16.194631
1001,2023-09-29,139.2,3.6,1.341395,12.370031
232,2021-08-21,113.0,1.3,-0.880795,9.978515
1177,2024-03-23,93.0,3.2,0.954927,8.152930
298,2021-10-26,83.8,2.7,0.471842,7.313161
233,2021-08-22,67.8,2.3,0.085375,5.852693
848,2023-04-29,62.5,3.8,1.534629,5.368913
849,2023-04-30,58.9,3.1,0.858310,5.040307
1325,2024-08-18,58.2,1.4,-0.784178,4.976412
188,2021-07-08,57.7,2.2,-0.011242,4.930772


In [22]:
storms_df.shape

(111, 23)

In [23]:
wdf.loc[wdf['precipitation'].idxmax()]

date              2021-09-01 00:00:00
wind_speed                        3.8
precipitation                   181.1
snowfall                          0.0
temp_max                         25.6
temp_min                         17.2
wt_fog                            1.0
wt_smoke_haze                     0.0
wt_thunder                        1.0
wt_heavy_fog                      1.0
wt_ice_pellets                    0.0
wt_glaze                          0.0
wt_hail                           0.0
wt_rain                           0.0
wt_snow                           0.0
wt_drizzle                        0.0
wt_mist                           0.0
wt_high_winds                     0.0
month                               9
year                             2021
month_name                        Sep
Name: 243, dtype: object

After z-score detection, visualise it

In [24]:
brush = alt.selection_interval(bind='scales')

# plot of storms detected (2021-2024)
storms_visual = alt.Chart(storms_df).transform_calculate( 
    is_thunder="datum.wt_thunder == 1 ? 'Yes' : 'No'"
).mark_circle(size=100, opacity=0.8).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('precipitation:Q', title='Daily Precipitation (mm)'),
    size=alt.Size('wind_z:Q', title='Wind Severity', scale=alt.Scale(range=[50, 300])),
    color=alt.Color('precip_z:Q', title='Rain Severity', scale=alt.Scale(scheme='reds')),
    tooltip=[
        alt.Tooltip('date:T', title='Date'),
        alt.Tooltip('precipitation:Q', format='.1f', title='Rainfall (mm)'),
        alt.Tooltip('wind_speed:Q', format='.1f', title='Wind Speed (m/s)'),
        alt.Tooltip('precip_z:Q', format='.2f', title='Rain Severity'),
        alt.Tooltip('wind_z:Q', format='.2f', title='Wind Severity'),
        alt.Tooltip('is_thunder:N', title='Thunderstorm') 
    ]
).properties(
    title={
        "text": "Storm/Flood Anomalies in NYC (2021-2024)",
        "subtitle": "Extreme Rainfall/Wind Events - Z-Score > 2", 
        "fontSize": 15,
    },
    width=750,
    height=350
).add_params(brush)

# hurricane Ida
ida_label = alt.Chart(pd.DataFrame({'date': ['2021-09-01'], 'precipitation': [181.1]})).mark_text(
    align='left', dx=10, dy=-10, fontSize=12, fontWeight='bold', color='red'
).encode(
    x='date:T',
    y='precipitation:Q',
    text=alt.value('Hurricane Ida (Flood Spike)')
)

#metropolitan area 
metro_label = alt.Chart(pd.DataFrame({'date': ['2023-09-29'], 'precipitation': [139.2]})).mark_text(
    align='left', dx=10, dy=-10, fontSize=12, fontWeight='bold', color='red'
).encode(
    x='date:T',
    y='precipitation:Q',
    text=alt.value('NYC Metro Flood')
)

# hurricane Henri
henri_label = alt.Chart(pd.DataFrame({'date': ['2021-08-21'], 'precipitation': [113.0]})).mark_text(
    align='left', dx=10, dy=-10, fontSize=12, fontWeight='bold', color='red'
).encode(
    x='date:T',
    y='precipitation:Q',
    text=alt.value('Hurricane Henri')
)

# "super soaker storm
soaker_label = alt.Chart(pd.DataFrame({'date': ['2024-03-23'], 'precipitation': [93.0]})).mark_text(
    align='left', dx=10, dy=-10, fontSize=12, fontWeight='bold', color='red'
).encode(
    x='date:T',
    y='precipitation:Q',
    text=alt.value('Super Soaker Storm')
)

storms_visual + ida_label + metro_label + henri_label + soaker_label

alt.LayerChart(...)

The anomaly analysis identifies several significant meteorological events between 2021 and 2024 where Precipitation or Wind speeds exceeded 2 standard deviations from the norm. Based on the graph, Hurricane Ida (Sept 2021) appears as the most extreme outlier with a record-breaking rainfall of over 180mm. Other identified anomalies such as Hurricane Henri (Aug 2021), NYC Metro Flood (Sep 2023), and Super Soaker Storm (Mar 2024). These events suggest that frequency of high-impact precipitation and wind anomalies is a notable feature of New York City’s recent climate profile.

<h2>Relationship between Seasonal Temperature and Electricity Demand</h2>

Fetch New York City's electricity hourly load from a local MongoDB database, retrieving only the required columns into a pandas DataFrame to reduce memory usage.
Energy data is imported from New York ISO.

Data Source (Downloaded): https://www.nyiso.com/custom-reports?report=load_forecast <br/>

In [25]:
try:   
    client = MongoClient("mongodb://localhost:27017/")

    db = client["cst_4068"]
    energy = db["nyiso_energy"]  
    
    # fetch date, hour, and NYC electricity load columns only
    query = {}
    projection = {
        "Local Date": 1, 
        "Hour Number": 1,
        "J - New York City Actual Load (MW)": 1,
        "_id": 0
    }
    e_results = energy.find(query, projection)  
    edf = pd.DataFrame(list(e_results))      
    
    # rename columns 
    rename_map = {
        'Local Date': 'date',
        'Hour Number': 'hour',
        'J - New York City Actual Load (MW)': 'electricity_load_mw'
    }

    edf = edf.rename(columns=rename_map)
    print(edf.head(50))
except Exception as e:
    print(f"An error occurred: {e}")

finally:
    if 'client' in locals():
        client.close()

        date  hour  electricity_load_mw
0   1/1/2023     1            4579.6137
1   1/1/2023     2            4429.6009
2   1/1/2023     3            4289.3015
3   1/1/2023     4            4175.7152
4   1/1/2023     5            4109.7769
5   1/1/2023     6            4098.9189
6   1/1/2023     7            4169.3338
7   1/1/2023     8            4234.6515
8   1/1/2023     9            4322.7212
9   1/1/2023    10            4417.0680
10  1/1/2023    11            4513.7877
11  1/1/2023    12            4613.3030
12  1/1/2023    13            4672.2943
13  1/1/2023    14            4719.3592
14  1/1/2023    15            4764.8764
15  1/1/2023    16            4823.5132
16  1/1/2023    17            4962.4498
17  1/1/2023    18            5146.2279
18  1/1/2023    19            5170.9267
19  1/1/2023    20            5123.5272
20  1/1/2023    21            5052.7346
21  1/1/2023    22            4955.9707
22  1/1/2023    23            4798.0386
23  1/1/2023    24            4583.1266


In [26]:
edf['date'] = pd.to_datetime(edf['date'], dayfirst=True)
edf['date'].dt.year.unique()

array([2023, 2024, 2022], dtype=int32)

Merge energy dataframe (data include years 2022, 2023, 2024) and weather dataframe. In this case, unit of hourly energy load switch to daily peak load.

In [27]:
# select required attributes only  
weather = wdf.copy()
weather = weather[['date', 'temp_max', 'temp_min', 'wind_speed']]

# create season column based on month
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:  # 9, 10, 11
        return 'Autumn'

weather['season'] = weather['date'].dt.month.apply(get_season)

# change hourly energy data to daily peak load
electricity_load = edf.groupby('date')['electricity_load_mw'].max().reset_index()
electricity_load.columns = ['date', 'peak_load_mw']

# merge weather and energy data on date
mdf = pd.merge(electricity_load, weather, on='date', how='inner')

mdf.sample(20)

,date,peak_load_mw,temp_max,temp_min,wind_speed,season
176,2022-06-26,7396.0574,31.7,23.3,1.60,Summer
494,2023-05-10,5678.7549,22.8,10.0,0.90,Spring
837,2024-06-22,8726.6743,33.9,23.9,0.40,Summer
819,2024-06-04,7601.5613,26.1,18.3,1.30,Summer
1007,2024-12-09,6260.1539,10.6,6.7,1.60,Winter
417,2023-02-22,6039.9785,6.7,2.8,2.70,Winter
37,2022-02-07,6610.7901,2.8,-1.0,1.60,Winter
529,2023-06-14,6802.5405,24.4,17.2,1.70,Summer
217,2022-08-06,8909.9702,33.3,25.0,1.80,Summer
848,2024-07-03,7334.0158,28.3,20.6,1.70,Summer


Visualise relationship between seasonal temperature and NYC's electricity demand  

In [28]:
# base chart
base = alt.Chart(mdf).encode(
    x=alt.X('temp_max:Q', title='Temperature (°C)'),
    y=alt.Y('peak_load_mw:Q', title='Peak Load (MW)'),
    tooltip=['temp_max', 'peak_load_mw', 'season']
).properties(
    width=500,
    height=350,
    title="Relationship Between Temperature and NYC's Electricity Demand" 
)

# add points with season color encoding
points = base.mark_point(
    filled=True,
    opacity=0.6,
    size=70
).encode(
    color=alt.Color('season:N', title="Season")
)

# smooth trend line 
trend = base.transform_loess('temp_max', 'peak_load_mw').mark_line(
    color='red',
    strokeWidth=3
)

points + trend

alt.LayerChart(...)

This scatter plot illustrates a non-linear, `U-shaped` relationship between daily temperature and New York City's peak electricity load. The colour coding indicates that summer (red) contributes to the highest demand levels, while winter (light green) produces a secondary increase in electricity usage.

As temperatures rise during the summer months, electricity demand increases sharply, reaching peaks of over 10,000 MW, largely due to air conditioning use. Conversely, as temperatures fall below 15°C, demand increases significantly, reflecting additional energy consumption for heating and lighting.

The lowest electricity demand is observed within a moderate temperature range (approximately 12°C to 20°C), where neither heating nor cooling demand is dominant. This results in a relatively stable usage of electricy around 5,500 MW during the spring and autumn.

While the overall relationship exhibits a `U-shaped` pattern, the relationship between each seasonal temperature and energy load is examined further below.

Regression Line and a Seasonal Dropdown Plot

In [29]:
# prepare dropdown options for season selection
# default select 'Summer'
season_options = ['Summer', 'Winter', 'Spring', 'Autumn']
selection = alt.selection_point(
    fields=['season'], 
    bind=alt.binding_select(options=season_options, name='Select Season: '), 
    value='Summer'
)

# base chart with selection filter
base = alt.Chart(mdf).encode(
    x=alt.X('temp_max:Q', title='Temperature (°C)', scale=alt.Scale(domain=[-10, 40])),
    y=alt.Y('peak_load_mw:Q', title='Peak Load (MW)', scale=alt.Scale(zero=False)),
    tooltip=['date:T', 'temp_max:Q', 'peak_load_mw:Q', 'season:N']
).add_params(
    selection
).transform_filter(
    selection
).properties(
    width=500,
    height=350,    
    title={
        "text": "Relationship Seasonal Temperature and Electricity Demand Analysis",      
        "subtitle": "(2022-2024)",
        "fontSize": 15        
    }
)

# add points with season color encoding
data_points = base.mark_point(
    filled=True,
    opacity=0.6,
    size=70
).encode(
    color=alt.Color('season:N', 
        scale=alt.Scale(
            domain=['Summer', 'Winter', 'Spring', 'Autumn'],
            range=["#ffa500", "#a0e6ec", "#bbcc16", "#f79762"] 
        ),
        legend=None # hide legend
    ) 
)
# draw regression line
reg_line = base.transform_regression(
    'temp_max', 'peak_load_mw'
).mark_line(
    color='red',
    strokeWidth=3
)

data_points + reg_line

alt.LayerChart(...)

<b>Summer (Strong Positive Correlation)</b></br>
A strong positive relationship is observed between approximately 20°C and 35°C. The regression line is steep, showing that as temperature rises, electricity load surges rapidly from 6,000 MW to 11,000 MW. This is primarily driven by air conditioning usage.

<b>Winter (Significant Negative Correlation)</b></br>
Between 15°C and -10°C, there is a significant negative correlation. As temperatures drop, electricity demand increases to approximately 7,200 MW due to heating requirements and shorter daylight hours requiring more lighting.

<b>Spring & Autumn (Weak Correlation)</b></br> 
During spring and autumn, the relationship is weaker. Data points are more dispersed and the regression line is relatively flat. Most of the days fall within a moderate temperature range (approximately 12°C to 20°C), where electricity demand remains relatively stable at around 5,500 MW, as neither cooling nor intensive heating is basically required.

Overall, the seasonal patterns explain the `U-shaped` relationship observed previously, with higher electricity demand occurring at both low and high temperature extremes. 
This indicates that electricity demand in NYC is more sensitive to temperature during periods of extreme heat and cold.

<h2>Comparing the weather conditions (1990s and 2020s)</h2>

Weather conditions are analysed by comparing two eras 1990s and the 2020s.

First of all, review the year.

In [30]:
wdf['date'].dt.year.unique()

array([2021, 2022, 2023, 2024, 1996, 1997, 1998, 1999], dtype=int32)

Create `era` attribute to distinguish 1990s and 2020s

In [31]:
wdf['era'] = ''
wdf.loc[wdf['year'] < 2000, 'era'] = '1990s'
wdf.loc[wdf['year'] >= 2020, 'era'] = '2020s'

Create `weather_event` attribute for different types of weather 

In [32]:
def get_weather_signature(row):
    if row['wt_thunder'] == 1: return 'Thunderstorm'
    if row['snowfall'] > 0 or row['wt_snow'] == 1: return 'Snow'
    if row['precipitation'] > 0.5 or row['wt_rain'] == 1 or row['wt_drizzle'] == 1: return 'Rain'
    if row['wt_fog'] == 1 or row['wt_heavy_fog'] == 1 or row['wt_mist'] == 1: return 'Fog/Mist'
    if row['wt_smoke_haze'] == 1: return 'Smoke/Haze'
    if row['wind_speed'] > 5.0 or row['wt_high_winds'] == 1: return 'High Winds'
    if row['wt_ice_pellets'] == 1 or row['wt_hail'] == 1: return 'Pellets/Hail'
    return 'Clear/Cloudy'

wdf['weather_event'] = wdf.apply(get_weather_signature, axis=1)
wdf.sample(10)

,date,wind_speed,precipitation,snowfall,temp_max,temp_min,wt_fog,wt_smoke_haze,wt_thunder,wt_heavy_fog,wt_ice_pellets,wt_glaze,wt_hail,wt_rain,wt_snow,wt_drizzle,wt_mist,wt_high_winds,month,year,month_name,era,weather_event
634,2022-09-27,1.9,0.0,0.0,21.1,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9,2022,Sep,2020s,Clear/Cloudy
2665,1999-04-19,2.7,0.3,0.0,15.6,7.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,4,1999,Apr,1990s,Rain
865,2023-05-16,2.3,0.0,0.0,26.1,14.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,2023,May,2020s,Clear/Cloudy
2463,1998-09-29,2.0,0.0,0.0,21.7,11.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9,1998,Sep,1990s,Clear/Cloudy
2620,1999-03-05,3.3,0.0,0.0,6.1,-1.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,1999,Mar,1990s,Clear/Cloudy
1148,2024-02-23,1.3,1.0,0.0,8.3,5.6,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,2024,Feb,2020s,Rain
697,2022-11-29,1.3,0.0,0.0,6.7,2.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11,2022,Nov,2020s,Clear/Cloudy
1106,2024-01-12,3.1,2.0,0.0,10.0,4.4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,2024,Jan,2020s,Rain
1310,2024-08-03,1.8,18.5,0.0,33.9,22.8,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8,2024,Aug,2020s,Thunderstorm
466,2022-04-12,1.7,1.8,0.0,22.2,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,2022,Apr,2020s,Rain


- Find each era total days
- Group weather conditions by era to obtain days of each weather condition 
- Find percentage of each weather condition

In [33]:
era_totals = wdf['era'].value_counts().reset_index(name='total_days')
era_totals.columns = ['era', 'total_days']

weather_summary = wdf.groupby(['era', 'weather_event']).size().reset_index(name='days')
weather_summary = weather_summary.merge(era_totals, on='era')

weather_summary['percentage'] = (weather_summary['days'] / weather_summary['total_days']) * 100

Total days of each era

In [34]:
era_totals

,era,total_days
0,2020s,1461
1,1990s,1461


Dataframe after finding each era total days and percentage value for each weather condition 

In [35]:
weather_summary

,era,weather_event,days,total_days,percentage
0,1990s,Clear/Cloudy,576,1461,39.425051
1,1990s,Fog/Mist,129,1461,8.829569
2,1990s,High Winds,30,1461,2.053388
3,1990s,Rain,472,1461,32.306639
4,1990s,Smoke/Haze,68,1461,4.654346
5,1990s,Snow,96,1461,6.570842
6,1990s,Thunderstorm,90,1461,6.160164
7,2020s,Clear/Cloudy,807,1461,55.236140
8,2020s,Fog/Mist,126,1461,8.624230
9,2020s,High Winds,12,1461,0.821355


Visualise the weather conditions by comparing 1990s and 2020s era

In [36]:
alt.Chart(weather_summary).mark_bar().encode(
    x=alt.X('weather_event:N', 
            title='Weather Events', 
            sort=alt.EncodingSortField(field='percentage', op='sum', order='descending'), 
            axis=alt.Axis(labelAngle=0)),       
    y=alt.Y('percentage:Q', title='Frequency (% of Days)'),         
    color=alt.Color('era:N', title='Era'), 
    xOffset='era:N', 
    tooltip=[
        alt.Tooltip('weather_event:N', title='Weather Event'),
        alt.Tooltip('era:N', title='Era'),
        alt.Tooltip('percentage:Q', format='.2f', title='Percentage')]  
).properties(
    width=800,
    height=350, 
    padding={'left': 10, 'top': 10, 'right': 10, 'bottom': 10},   
    title={
        "text": "Comparing Weather Conditions (1990s vs 2020s)",      
        "fontSize": 15
    }
).display()

alt.Chart(...)

The comparison suggests a shift towards more frequent dry or cloudy conditions and fewer rain and snow events, with a slight increase in thunderstorms and hail. Foggy conditions remains relative stable. Further details are discussed in the storytelling section.

<small>(Thunderstorm frequency for the 1990s is recorded as zero due to historical gap; therefore, the thunderstorm attribute was preprocessed in the earlier section to ensure it is useful in this graph.)</small>

<hr>
<h1>Storytelling / Presentation</h1>

<b>Context:</b> NYC's seasonal identity is shifting. The winter of the past is being changed, with fewer snow events, and rain frequency drop significantly. 

<b>Data:</b> Weather condition frequencies are compared between the 1990s and the 2020s.

<b>Task:</b> Understanding how NYC's traditional pattern of steady rainfall has changed over time and how extreme events have evolved.

<b>The Era of Steady Rhythms</b></br>
<p><i><font color="#3681a4">In the 1990s, NYC had a clear, predictable seasonal identity. It was a city of regular precipitation. Roughly 32% of days experienced rain, and snow was a regular winter feature, occurring on nearly 7% of days. This represents the historical baseline - a city where rain and snow were expected parts of the annual cycle.</font></i></p>

<img src="img/nyc_1990s.svg">

<b>The Shift of Dry Weather</b><br/>
<p><i><font color="#3681a4">After 25 years later the weather began to change. The 'Clear/Cloudy' frequency surged to 55%, while the occurrence of rain has declined noticeably. Most notably, winters are becoming less prominent, with snow frequency dropping to nearly 2%. NYC is becoming drier and sunnier on average</font></i></p>

<img src="img/nyc_2020s.svg">

<b>The hiding Paradox</b><br/>
<p><i><font color="#3681a4">However, this increase in sunny days hides a dangerous paradox. Despite fewer rainy days overall in the 2020s, the atmosphere appears more energetic. Thunderstorms and Hail events are now more frequent than they were 25 years ago.</font></i></p>

<img src="img/ts_pellets.svg">

<b>Rain Is Less Frequent - But More Intense</b><br/>
<p><i><font color="#3681a4">Although the frequency of raining dropped, the nature of that rain has shifted from steady to spiky. The 2020s era have more intense bursts of rain. This helps explain why flooding can still occur, even in a drier period overall.</font></i></p>

<img src="img/frequent_to_intense.svg">

As a conclusion, New York City has transitioned from a stable 1990s baseline to a highly volatile climate in the 2020s. While routine precipitation and snowfall have decreased, this dry and sunnier trend masks a dangerous surge in thunderstorms. Ultimately, NYC has shifted from weather frequency to intensity, indicating that traditional daily averages are insufficient for modern urban risk assessment and infrastructure planning.

<div style="text-align:center;">
Created By: Shwe <br/>
</div>